# Reliability Diagrams and Probability Reporting

This notebook documents probability-quality reporting for the final models. It reconstructs classwise reliability bins, compares raw and calibrated XGBoost probabilities, and produces aggregate English-labelled figures without refitting or selecting a model.


## Release Boundary

This restricted code-inspection copy retains the analytical logic but does not include Banjarbaru review data, row-level predictions, manual-audit records, model checkpoints, or private run artifacts. The restricted path placeholder documents the authorized environment and is not an executable public-data contract.


### Configure the reliability-reporting contract

Imports reporting and plotting utilities, defines the restricted project-root placeholder, identifies repeated-seed prediction and metric inputs, and fixes the class order, bin count, and expected seeds.


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import uuid

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd


PROJECT_ROOT = Path('RESTRICTED_PROJECT_ROOT')
ROUND1_DIR = PROJECT_ROOT / 'journal_round1'
R1_08_ROOT = ROUND1_DIR / 'artifacts' / 'r1_08_repeated_seed_evaluation_v1'
R1_07_ROOT = ROUND1_DIR / 'artifacts' / 'r1_07_revised_stacking_calibration_v1'
OUTPUT_DIR = ROUND1_DIR / 'reports' / 'reliability_probability_reporting' / 'r1_08_repeated_seed'
FIGURE_DIR = ROUND1_DIR / 'figures' / 'reliability_probability_reporting' / 'r1_08_repeated_seed'



EXPECTED_SEEDS = [42, 52, 62]
LABEL_ORDER = ['negatif', 'netral', 'positif']
N_CLASSES = len(LABEL_ORDER)
N_BINS = 10
EPSILON = 1e-12

PREDICTION_PATH = R1_08_ROOT / 'locked_test_predictions_per_model_long_with_silver.csv'
METRIC_PATH = R1_08_ROOT / 'locked_test_silver_metrics_per_seed.csv'
CONTRACT_PATH = R1_08_ROOT / 'prediction_contract.json'
MANIFEST_PATH = R1_08_ROOT / 'run_manifest.json'
STATE_PATH = R1_08_ROOT / 'run_state.json'


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    frame.to_csv(temporary, index=False, encoding='utf-8')
    os.replace(temporary, path)


def atomic_write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=True, default=float), encoding='utf-8')
    os.replace(temporary, path)


def validate_probability_matrix(values: np.ndarray, name: str) -> None:
    values = np.asarray(values, dtype=float)
    if values.ndim != 2 or values.shape[1] != N_CLASSES:
        raise ValueError(f'{name}: invalid probability shape {values.shape}.')
    if not np.isfinite(values).all() or values.min() < -1e-8 or values.max() > 1 + 1e-8:
        raise ValueError(f'{name}: invalid probability values.')
    if not np.allclose(values.sum(axis=1), 1.0, atol=1e-5):
        raise ValueError(f'{name}: probability rows do not sum to one.')


def summarize(values: np.ndarray) -> dict:
    values = np.asarray(values, dtype=float)
    return {
        'n': int(len(values)), 'mean': float(values.mean()), 'sample_sd': float(values.std(ddof=1)),
        'p05': float(np.quantile(values, 0.05)), 'p50': float(np.quantile(values, 0.50)),
        'p95': float(np.quantile(values, 0.95)), 'minimum': float(values.min()), 'maximum': float(values.max()),
    }


def classwise_reliability_rows(seed: int, model_key: str, model_name: str, probabilities: np.ndarray, y_true: np.ndarray) -> tuple[list[dict], list[dict]]:
    bin_rows, ece_rows = [], []
    for class_id, class_label in enumerate(LABEL_ORDER):
        probability = probabilities[:, class_id]
        event = (y_true == class_id).astype(float)
        bin_id = np.minimum((probability * N_BINS).astype(int), N_BINS - 1)
        class_ece = 0.0
        for current_bin in range(N_BINS):
            mask = bin_id == current_bin
            n_bin = int(mask.sum())
            left, right = current_bin / N_BINS, (current_bin + 1) / N_BINS
            if n_bin:
                mean_probability = float(probability[mask].mean())
                observed_frequency = float(event[mask].mean())
                absolute_gap = abs(observed_frequency - mean_probability)
                weighted_gap = float((n_bin / len(probability)) * absolute_gap)
                class_ece += weighted_gap
            else:
                mean_probability = np.nan
                observed_frequency = np.nan
                absolute_gap = np.nan
                weighted_gap = 0.0
            bin_rows.append({
                'seed': seed, 'model_key': model_key, 'model': model_name, 'class_id': class_id, 'class_label': class_label,
                'bin_index': current_bin, 'bin_left': left, 'bin_right': right, 'n_bin': n_bin, 'bin_coverage': n_bin / len(probability),
                'mean_predicted_probability': mean_probability, 'observed_class_frequency': observed_frequency,
                'absolute_gap': absolute_gap, 'weighted_absolute_gap': weighted_gap,
            })
        ece_rows.append({
            'seed': seed, 'model_key': model_key, 'model': model_name, 'class_id': class_id,
            'class_label': class_label, 'classwise_ece_component': class_ece,
        })
    return bin_rows, ece_rows


print('Repeated-seed evaluation input:', R1_08_ROOT)
print('Output reports:', OUTPUT_DIR)
print('Output figures:', FIGURE_DIR)


### Validate probability inputs and reconstruct reliability tables

Checks completed repeated-seed evaluation artifacts, contracts, checksums, model order, and probability matrices. It then reconstructs classwise equal-width reliability bins and verifies the reported ECE calculation.


In [ ]:
for path in [PREDICTION_PATH, METRIC_PATH, CONTRACT_PATH, MANIFEST_PATH, STATE_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

contract = json.loads(CONTRACT_PATH.read_text(encoding='utf-8'))
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
state = json.loads(STATE_PATH.read_text(encoding='utf-8'))
if state.get('status') != 'completed' or manifest.get('locked_test_metrics_computed') is not True:
    raise RuntimeError('Repeated-seed evaluation is not completed.')
if contract.get('seeds') != EXPECTED_SEEDS or manifest.get('protocol', {}).get('planned_seeds') != EXPECTED_SEEDS:
    raise RuntimeError('Repeated-seed seed contract does not match the fixed reliability-reporting contract.')
declared_label_source = manifest.get('protocol', {}).get('locked_test_evaluation_label')
locked_test_target = str(manifest.get('evaluation_target', '')).lower()
if declared_label_source != 'LSTM-derived silver labels only' or 'lstm-derived silver labels' not in locked_test_target or 'locked test' not in locked_test_target:
    raise RuntimeError('Unexpected locked-test evaluation target semantics.')
if manifest.get('output_checksums', {}).get(PREDICTION_PATH.name) != sha256_file(PREDICTION_PATH):
    raise RuntimeError('Repeated-seed prediction checksum differs from the manifest.')
if manifest.get('output_checksums', {}).get(METRIC_PATH.name) != sha256_file(METRIC_PATH):
    raise RuntimeError('Repeated-seed metric checksum differs from the manifest.')

MODEL_ORDER = [(item['model_key'], item['model']) for item in contract.get('model_order', [])]
if len(MODEL_ORDER) != 5 or [key for key, _ in MODEL_ORDER] != ['svm', 'indobert', 'wa_svm_bert', 'xgboost', 'calibrated_xgboost']:
    raise RuntimeError('Unexpected repeated-seed final-model contract.')
MODEL_NAME = dict(MODEL_ORDER)

predictions = pd.read_csv(PREDICTION_PATH)
metrics = pd.read_csv(METRIC_PATH)
required_prediction_columns = {
    'seed', 'row_id', 'y_silver', 'y_silver_label', 'model_key', 'model',
    'prob_negatif', 'prob_netral', 'prob_positif', 'prediction', 'prediction_label', 'confidence', 'entropy',
}
required_metric_columns = {'seed', 'split', 'model_key', 'model', 'probability_source', 'n', 'accuracy', 'macro_f1', 'ece', 'brier_score'}
if missing := sorted(required_prediction_columns.difference(predictions.columns)):
    raise ValueError(f'Prediction columns missing: {missing}')
if missing := sorted(required_metric_columns.difference(metrics.columns)):
    raise ValueError(f'Metric columns missing: {missing}')
if sorted(predictions['seed'].unique().tolist()) != EXPECTED_SEEDS or set(predictions['model_key']) != set(MODEL_NAME):
    raise RuntimeError('Unexpected seed or model in final prediction artifact.')
if len(predictions) != 1172 * len(EXPECTED_SEEDS) * len(MODEL_ORDER) or predictions.duplicated(['seed', 'row_id', 'model_key']).any():
    raise RuntimeError('Final prediction keys or row count are invalid.')
if not metrics['split'].eq('locked_test_silver').all() or len(metrics) != len(EXPECTED_SEEDS) * len(MODEL_ORDER):
    raise RuntimeError('Metric table does not contain the expected locked-test rows.')

bin_records, class_ece_records, class_summary_records, confidence_records = [], [], [], []
for (seed, model_key), group in predictions.groupby(['seed', 'model_key'], sort=True):
    group = group.sort_values('row_id').reset_index(drop=True)
    model_name = MODEL_NAME[model_key]
    if not group['model'].eq(model_name).all() or len(group) != 1172:
        raise RuntimeError(f'{seed}/{model_key}: model name or row count is invalid.')
    probabilities = group[['prob_negatif', 'prob_netral', 'prob_positif']].to_numpy(float)
    y_true = group['y_silver'].to_numpy(int)
    validate_probability_matrix(probabilities, f'{seed}/{model_key}')
    if not np.isin(y_true, np.arange(N_CLASSES)).all():
        raise RuntimeError(f'{seed}/{model_key}: silver label is outside the expected range.')
    if not np.array_equal(group['prediction'].to_numpy(int), probabilities.argmax(axis=1)):
        raise RuntimeError(f'{seed}/{model_key}: prediction differs from probability argmax.')
    if not np.allclose(group['confidence'].to_numpy(float), probabilities.max(axis=1), atol=1e-8):
        raise RuntimeError(f'{seed}/{model_key}: confidence differs from probability maximum.')

    current_bins, current_ece = classwise_reliability_rows(seed, model_key, model_name, probabilities, y_true)
    bin_records.extend(current_bins)
    class_ece_records.extend(current_ece)
    for class_id, class_label in enumerate(LABEL_ORDER):
        summary = summarize(probabilities[:, class_id])
        summary.update({
            'seed': seed, 'model_key': model_key, 'model': model_name, 'class_id': class_id, 'class_label': class_label,
            'silver_class_prevalence': float((y_true == class_id).mean()),
            'hard_prediction_share': float((probabilities.argmax(axis=1) == class_id).mean()),
        })
        class_summary_records.append(summary)
    confidence_summary = summarize(group['confidence'].to_numpy(float))
    confidence_summary.update({
        'seed': seed, 'model_key': model_key, 'model': model_name,
        'mean_entropy': float(group['entropy'].mean()), 'sample_sd_entropy': float(group['entropy'].std(ddof=1)),
    })
    confidence_records.append(confidence_summary)

reliability_bins = pd.DataFrame(bin_records).sort_values(['seed', 'model_key', 'class_id', 'bin_index']).reset_index(drop=True)
classwise_ece = pd.DataFrame(class_ece_records).sort_values(['seed', 'model_key', 'class_id']).reset_index(drop=True)
class_probability_summary = pd.DataFrame(class_summary_records).sort_values(['seed', 'model_key', 'class_id']).reset_index(drop=True)
confidence_summary = pd.DataFrame(confidence_records).sort_values(['seed', 'model_key']).reset_index(drop=True)

reconstructed_ece = classwise_ece.groupby(['seed', 'model_key', 'model'], as_index=False).agg(
    reconstructed_classwise_equal_width_ece=('classwise_ece_component', 'mean')
)
metric_reference = metrics[['seed', 'model_key', 'model', 'probability_source', 'accuracy', 'macro_f1', 'ece', 'brier_score']].copy()
ece_check = reconstructed_ece.merge(metric_reference, on=['seed', 'model_key', 'model'], how='inner', validate='one_to_one')
ece_check['ece_difference_reconstructed_minus_r1_08'] = ece_check['reconstructed_classwise_equal_width_ece'] - ece_check['ece']
if not np.allclose(ece_check['ece_difference_reconstructed_minus_r1_08'], 0.0, atol=1e-10):
    raise RuntimeError('Reliability-bin ECE cannot reproduce the repeated-seed classwise ECE.')

print('Validated final prediction rows:', len(predictions))
print('Reliability-bin rows:', len(reliability_bins))
print('ECE reconstruction passed for all seed-model combinations.')


### Quantify raw versus calibrated probability changes

Compares XGBoost and Calibrated XGBoost metrics and probability summaries per seed, including the validation cross-fitting information used to document the calibration procedure.


In [ ]:
raw_cal_metrics = metric_reference.loc[metric_reference['model_key'].isin(['xgboost', 'calibrated_xgboost'])].copy()
metric_wide = raw_cal_metrics.pivot(index='seed', columns='model_key', values=['accuracy', 'macro_f1', 'ece', 'brier_score'])
calibration_effect_rows = []
for seed in EXPECTED_SEEDS:
    record = {'seed': seed}
    for metric_name in ['accuracy', 'macro_f1', 'ece', 'brier_score']:
        raw_value = float(metric_wide.loc[seed, (metric_name, 'xgboost')])
        calibrated_value = float(metric_wide.loc[seed, (metric_name, 'calibrated_xgboost')])
        record[f'raw_xgboost_{metric_name}'] = raw_value
        record[f'calibrated_xgboost_{metric_name}'] = calibrated_value
        record[f'calibrated_minus_raw_{metric_name}'] = calibrated_value - raw_value
        record[f'raw_minus_calibrated_{metric_name}'] = raw_value - calibrated_value
    calibration_effect_rows.append(record)
calibration_effect = pd.DataFrame(calibration_effect_rows).sort_values('seed').reset_index(drop=True)

calibration_effect_summary_rows = []
for metric_name in ['accuracy', 'macro_f1', 'ece', 'brier_score']:
    delta = calibration_effect[f'calibrated_minus_raw_{metric_name}'].to_numpy(float)
    calibration_effect_summary_rows.append({
        'metric': metric_name, 'direction': 'calibrated_minus_raw', 'n_seeds': len(delta),
        'mean_delta': float(delta.mean()), 'sample_sd_delta': float(delta.std(ddof=1)),
        'minimum_delta': float(delta.min()), 'maximum_delta': float(delta.max()),
    })
calibration_effect_summary = pd.DataFrame(calibration_effect_summary_rows)

crossfit_audit_rows = []
for seed in EXPECTED_SEEDS:
    crossfit_path = R1_07_ROOT / f'seed_{seed}' / 'calibration_crossfit_summary.csv'
    if not crossfit_path.is_file():
        raise FileNotFoundError(crossfit_path)
    crossfit = pd.read_csv(crossfit_path)
    required_crossfit = {
        'calibration_fold', 'fit_n', 'eval_n', 'fit_unique_groups', 'eval_unique_groups', 'group_overlap',
        'raw_eval_ece', 'calibrated_eval_ece', 'raw_eval_brier', 'calibrated_eval_brier',
    }
    if missing := sorted(required_crossfit.difference(crossfit.columns)):
        raise ValueError(f'Seed {seed}: calibration summary columns missing: {missing}')
    if len(crossfit) != 5 or not crossfit['group_overlap'].eq(0).all():
        raise RuntimeError(f'Seed {seed}: cross-fitted calibration group guardrail failed.')
    crossfit_audit_rows.append({
        'seed': seed, 'calibration_folds': len(crossfit), 'all_group_overlap_zero': bool(crossfit['group_overlap'].eq(0).all()),
        'mean_raw_eval_ece': float(crossfit['raw_eval_ece'].mean()), 'mean_calibrated_eval_ece': float(crossfit['calibrated_eval_ece'].mean()),
        'mean_raw_eval_brier': float(crossfit['raw_eval_brier'].mean()), 'mean_calibrated_eval_brier': float(crossfit['calibrated_eval_brier'].mean()),
        'mean_raw_minus_calibrated_ece': float((crossfit['raw_eval_ece'] - crossfit['calibrated_eval_ece']).mean()),
        'mean_raw_minus_calibrated_brier': float((crossfit['raw_eval_brier'] - crossfit['calibrated_eval_brier']).mean()),
    })
crossfit_audit = pd.DataFrame(crossfit_audit_rows).sort_values('seed').reset_index(drop=True)

for filename, frame in {
    'classwise_reliability_bins_per_seed.csv': reliability_bins,
    'classwise_ece_reconstruction_per_seed.csv': ece_check,
    'probability_class_summary_per_seed.csv': class_probability_summary,
    'prediction_confidence_entropy_summary_per_seed.csv': confidence_summary,
    'calibration_effect_raw_vs_calibrated_xgboost_per_seed.csv': calibration_effect,
    'calibration_effect_raw_vs_calibrated_xgboost_repeated_seed_summary.csv': calibration_effect_summary,
    'calibration_crossfit_group_audit_per_seed.csv': crossfit_audit,
}.items():
    atomic_write_csv(OUTPUT_DIR / filename, frame)

print('Cross-fitted calibration audit:')
print(crossfit_audit.to_string(index=False))
print('Locked-test raw versus calibrated effect:')
print(calibration_effect.to_string(index=False))


### Generate raw-versus-calibrated reliability diagrams

Creates classwise reliability diagrams for XGBoost before and after calibration across the three seeds. The figure cell is retained because the final diagram supports probability reporting in the manuscript.


In [ ]:
curve_colors = {'xgboost': '#1f77b4', 'calibrated_xgboost': '#d62728'}
curve_labels = {'xgboost': 'XGBoost', 'calibrated_xgboost': 'Calibrated XGBoost'}
fig, axes = plt.subplots(len(EXPECTED_SEEDS), N_CLASSES, figsize=(14.0, 11.5), sharex=True, sharey=True)
for row, seed in enumerate(EXPECTED_SEEDS):
    for column, class_label in enumerate(LABEL_ORDER):
        axis = axes[row, column]
        axis.plot([0, 1], [0, 1], linestyle='--', color='#555555', linewidth=1.0, label='Ideal calibration')
        for model_key in ['xgboost', 'calibrated_xgboost']:
            curve = reliability_bins.loc[
                reliability_bins['seed'].eq(seed) & reliability_bins['model_key'].eq(model_key) &
                reliability_bins['class_label'].eq(class_label) & reliability_bins['n_bin'].gt(0)
            ].sort_values('bin_index')
            axis.plot(
                curve['mean_predicted_probability'], curve['observed_class_frequency'], marker='o', markersize=3.8,
                linewidth=1.6, color=curve_colors[model_key], label=curve_labels[model_key],
            )
        axis.set_xlim(0, 1)
        axis.set_ylim(0, 1)
        axis.grid(alpha=0.22)
        if row == 0:
            axis.set_title(f'P({class_label})', fontsize=12)
        if column == 0:
            axis.set_ylabel(f'Seed {seed}: observed frequency')
        if row == len(EXPECTED_SEEDS) - 1:
            axis.set_xlabel('Mean predicted probability')
handles = [
    Line2D([0], [0], color='#555555', linestyle='--', label='Ideal calibration'),
    Line2D([0], [0], color=curve_colors['xgboost'], marker='o', label='XGBoost'),
    Line2D([0], [0], color=curve_colors['calibrated_xgboost'], marker='o', label='Calibrated XGBoost'),
]
fig.suptitle('Classwise reliability diagrams: XGBoost before and after Platt calibration', fontsize=16, y=0.985)
fig.text(0.5, 0.953, 'Locked test; reference is LSTM-derived silver labels; 10 equal-width bins', ha='center', va='center', fontsize=10.5)
fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.01), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(left=0.09, right=0.99, bottom=0.10, top=0.90, wspace=0.12, hspace=0.22)
reliability_figure = FIGURE_DIR / 'reliability_diagram_xgboost_raw_vs_calibrated_per_seed.png'
fig.savefig(reliability_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

plot_data = calibration_effect.set_index('seed')
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.7))
for axis, metric_name, title in zip(axes, ['ece', 'brier_score'], ['Classwise ECE', 'Multiclass Brier score']):
    positions = np.arange(len(EXPECTED_SEEDS))
    width = 0.34
    axis.bar(positions - width / 2, plot_data.loc[EXPECTED_SEEDS, f'raw_xgboost_{metric_name}'], width, label='XGBoost', color=curve_colors['xgboost'])
    axis.bar(positions + width / 2, plot_data.loc[EXPECTED_SEEDS, f'calibrated_xgboost_{metric_name}'], width, label='Calibrated XGBoost', color=curve_colors['calibrated_xgboost'])
    axis.set_title(title)
    axis.set_xticks(positions, [f'Seed {seed}' for seed in EXPECTED_SEEDS])
    axis.set_ylabel('Lower is better')
    axis.grid(axis='y', alpha=0.22)
axes[0].legend(frameon=False)
fig.suptitle('Probability quality before and after calibration on the locked test', fontsize=14, y=0.98)
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.15, top=0.84, wspace=0.24)
calibration_effect_figure = FIGURE_DIR / 'calibration_effect_raw_vs_calibrated_xgboost_per_seed.png'
fig.savefig(calibration_effect_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

print('Saved:', reliability_figure)
print('Saved:', calibration_effect_figure)


### Write aggregate reporting tables and guardrails

Writes aggregate reliability-bin, ECE-reconstruction, probability-summary, and calibration-effect tables with a manifest that states no model fitting, threshold tuning, seed selection, or cross-seed probability averaging occurs here.


In [ ]:
required_outputs = [
    'classwise_reliability_bins_per_seed.csv',
    'classwise_ece_reconstruction_per_seed.csv',
    'probability_class_summary_per_seed.csv',
    'prediction_confidence_entropy_summary_per_seed.csv',
    'calibration_effect_raw_vs_calibrated_xgboost_per_seed.csv',
    'calibration_effect_raw_vs_calibrated_xgboost_repeated_seed_summary.csv',
    'calibration_crossfit_group_audit_per_seed.csv',
]
if len(reliability_bins) != len(EXPECTED_SEEDS) * len(MODEL_ORDER) * N_CLASSES * N_BINS:
    raise RuntimeError('Unexpected reliability-bin row count.')
if len(ece_check) != len(EXPECTED_SEEDS) * len(MODEL_ORDER):
    raise RuntimeError('Incomplete ECE reconstruction table.')
if not crossfit_audit['all_group_overlap_zero'].all():
    raise RuntimeError('Cross-fitted calibration audit is incomplete.')

output_manifest = {
    'status': 'completed',
    'completed_at_utc': utc_now(),
    'analysis_version': 'r1_09_reliability_probability_reporting_v1',
    'source_artifact': str(R1_08_ROOT.relative_to(PROJECT_ROOT)),
    'source_protocol_hash': manifest.get('protocol_hash'),
    'seeds': EXPECTED_SEEDS,
    'models': [dict(model_key=key, model=name) for key, name in MODEL_ORDER],
    'locked_test_reference': 'LSTM-derived silver labels only',
    'ece_definition': 'Classwise equal-width ECE with 10 bins, averaged over three classes; reconstructed and checked against repeated-seed evaluation.',
    'reliability_figure_scope': 'Raw XGBoost versus Calibrated XGBoost only because Platt calibration is applied only to XGBoost.',
    'guardrails': [
        'No training, calibration fitting, threshold tuning, seed selection, model selection, or probability averaging across seeds occurred in this reliability-reporting stage.',
        'Per-seed curves are retained; pooled curves are not created as a new model.',
        'Locked-test results are agreement with LSTM-derived silver labels, not full human-label performance.',
        'Reliability diagrams are descriptive evidence and do not establish model superiority by themselves.',
    ],
    'input_checksums': {PREDICTION_PATH.name: sha256_file(PREDICTION_PATH), METRIC_PATH.name: sha256_file(METRIC_PATH)},
    'output_checksums': {filename: sha256_file(OUTPUT_DIR / filename) for filename in required_outputs},
    'figure_checksums': {
        reliability_figure.name: sha256_file(reliability_figure),
        calibration_effect_figure.name: sha256_file(calibration_effect_figure),
    },
}
atomic_write_json(OUTPUT_DIR / 'r1_09_reliability_probability_reporting_manifest.json', output_manifest)
print('Reliability reporting completed. Manifest:', OUTPUT_DIR / 'r1_09_reliability_probability_reporting_manifest.json')


### Create English-labelled publication figures

Builds the release-ready reliability figure with English class labels and line-style legends. The final image is distributed separately through the approved aggregate-output whitelist.


In [ ]:
DISPLAY_CLASS_LABELS = {'negatif': 'negative', 'netral': 'neutral', 'positif': 'positive'}

fig, axes = plt.subplots(len(EXPECTED_SEEDS), N_CLASSES, figsize=(14.0, 11.5), sharex=True, sharey=True)
for row, seed in enumerate(EXPECTED_SEEDS):
    for column, class_label in enumerate(LABEL_ORDER):
        axis = axes[row, column]
        axis.plot([0, 1], [0, 1], linestyle='--', color='#555555', linewidth=1.0)
        for model_key in ['xgboost', 'calibrated_xgboost']:
            curve = reliability_bins.loc[
                reliability_bins['seed'].eq(seed) & reliability_bins['model_key'].eq(model_key) &
                reliability_bins['class_label'].eq(class_label) & reliability_bins['n_bin'].gt(0)
            ].sort_values('bin_index')
            axis.plot(
                curve['mean_predicted_probability'], curve['observed_class_frequency'],
                marker='o', markersize=3.8, linewidth=1.6, color=curve_colors[model_key],
            )
        axis.set_xlim(0, 1)
        axis.set_ylim(0, 1)
        axis.grid(alpha=0.22)
        if row == 0:
            axis.set_title(f"P({DISPLAY_CLASS_LABELS[class_label]})", fontsize=12)
        if column == 0:
            axis.set_ylabel(f'Seed {seed}: observed frequency')
        if row == len(EXPECTED_SEEDS) - 1:
            axis.set_xlabel('Mean predicted probability')
reliability_handles = [
    Line2D([0], [0], color='#555555', linestyle='--', label='Ideal calibration'),
    Line2D([0], [0], color=curve_colors['xgboost'], marker='o', label='XGBoost'),
    Line2D([0], [0], color=curve_colors['calibrated_xgboost'], marker='o', label='Calibrated XGBoost'),
]
fig.suptitle('Classwise reliability diagrams: XGBoost before and after Platt calibration', fontsize=16, y=0.985)
fig.text(0.5, 0.953, 'Locked test; reference is LSTM-derived silver labels; 10 equal-width bins', ha='center', va='center', fontsize=10.5)
fig.legend(handles=reliability_handles, loc='lower center', bbox_to_anchor=(0.5, 0.01), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(left=0.09, right=0.99, bottom=0.10, top=0.90, wspace=0.12, hspace=0.22)
reliability_figure = FIGURE_DIR / 'reliability_diagram_xgboost_raw_vs_calibrated_per_seed.png'
fig.savefig(reliability_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

plot_data = calibration_effect.set_index('seed')
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.7))
for axis, metric_name, title in zip(axes, ['ece', 'brier_score'], ['Classwise ECE', 'Multiclass Brier score']):
    positions = np.arange(len(EXPECTED_SEEDS))
    width = 0.34
    axis.bar(positions - width / 2, plot_data.loc[EXPECTED_SEEDS, f'raw_xgboost_{metric_name}'], width, label='XGBoost', color=curve_colors['xgboost'])
    axis.bar(positions + width / 2, plot_data.loc[EXPECTED_SEEDS, f'calibrated_xgboost_{metric_name}'], width, label='Calibrated XGBoost', color=curve_colors['calibrated_xgboost'])
    axis.set_title(title)
    axis.set_xticks(positions, [f'Seed {seed}' for seed in EXPECTED_SEEDS])
    axis.set_ylabel('Lower is better')
    axis.grid(axis='y', alpha=0.22)
axes[0].legend(frameon=False)
fig.suptitle('Probability quality before and after calibration on the locked test', fontsize=14, y=0.98)
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.15, top=0.84, wspace=0.24)
calibration_effect_figure = FIGURE_DIR / 'calibration_effect_raw_vs_calibrated_xgboost_per_seed.png'
fig.savefig(calibration_effect_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

manifest_path = OUTPUT_DIR / 'r1_09_reliability_probability_reporting_manifest.json'
publication_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
publication_manifest['publication_visualization'] = {
    'class_display_labels': DISPLAY_CLASS_LABELS,
    'main_manuscript_figure': str(reliability_figure.relative_to(PROJECT_ROOT)),
    'supplementary_figure_s1': str(calibration_effect_figure.relative_to(PROJECT_ROOT)),
    'note': 'Display labels are English-only rendering labels; CSV class labels remain unchanged.',
}
publication_manifest['figure_checksums'] = {
    reliability_figure.name: sha256_file(reliability_figure),
    calibration_effect_figure.name: sha256_file(calibration_effect_figure),
}
atomic_write_json(manifest_path, publication_manifest)
print('Publication-ready main figure:', reliability_figure)
print('Publication-ready Supplementary Figure S1:', calibration_effect_figure)
